# Problem Summary:
### NovaGen Research Labs collected health data from 9,800 individuals,
### including numerical and categorical features (e.g., medical, lifestyle, physiological data).
### They need a machine learning model to classify individuals as "healthy" or "unhealthy".
### This model will help in participant selection and risk-based analysis for research studies.

In [25]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Data Pre Processing

In [4]:
df = pd.read_csv("novagen_dataset.csv")

In [5]:
df

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type__Vegan,Diet_Type__Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,2.0,26.0,111.0,198.0,99.0,72.0,4.0,1.0,5.0,5.0,...,1,2,1,0,1,False,True,True,False,False
1,8.0,24.0,121.0,199.0,103.0,75.0,2.0,1.0,2.0,9.0,...,1,2,1,2,2,False,False,True,False,False
2,81.0,27.0,147.0,203.0,100.0,74.0,10.0,-0.0,5.0,1.0,...,2,0,0,1,0,True,False,False,False,False
3,25.0,21.0,150.0,199.0,102.0,70.0,7.0,3.0,3.0,3.0,...,1,2,1,2,0,True,False,False,True,False
4,24.0,26.0,146.0,202.0,99.0,76.0,10.0,2.0,5.0,1.0,...,2,0,2,0,2,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9544,5.0,22.0,109.0,203.0,98.0,75.0,8.0,1.0,6.0,0.0,...,0,2,2,1,0,True,False,True,False,False
9545,94.0,26.0,144.0,203.0,96.0,72.0,8.0,4.0,2.0,1.0,...,1,0,1,0,2,False,True,False,True,False
9546,10.0,23.0,185.0,198.0,103.0,72.0,4.0,5.0,5.0,6.0,...,1,0,2,0,1,True,False,True,False,False
9547,50.0,29.0,166.0,200.0,100.0,74.0,8.0,2.0,3.0,4.0,...,2,0,0,1,1,True,False,True,False,False


In [6]:
df.shape

(9549, 23)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9549 entries, 0 to 9548
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    9549 non-null   float64
 1   BMI                    9549 non-null   float64
 2   Blood_Pressure         9549 non-null   float64
 3   Cholesterol            9549 non-null   float64
 4   Glucose_Level          9549 non-null   float64
 5   Heart_Rate             9549 non-null   float64
 6   Sleep_Hours            9549 non-null   float64
 7   Exercise_Hours         9549 non-null   float64
 8   Water_Intake           9549 non-null   float64
 9   Stress_Level           9549 non-null   float64
 10  Target                 9549 non-null   int64  
 11  Smoking                9549 non-null   int64  
 12  Alcohol                9549 non-null   int64  
 13  Diet                   9549 non-null   int64  
 14  MentalHealth           9549 non-null   int64  
 15  Phys

# Baseline Model

In [8]:
x = df.drop("Target", axis=1)
y = df["Target"]

In [9]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

# Scaling

In [10]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [11]:
x_train_scaled_df = pd.DataFrame(x_train_scaled, columns=x_train.columns)
print(x_train_scaled_df)

           Age       BMI  Blood_Pressure  Cholesterol  Glucose_Level  \
0     0.888944  3.268180        1.788624     0.461629      -0.098087   
1     1.211825  0.172180        0.094311     0.967494      -0.560609   
2     1.575067 -2.407819        0.671099     0.967494       0.826958   
3     0.606423 -0.859820        0.599000    -1.055966       0.364435   
4    -0.402582  0.172180        0.743197    -1.055966      -0.098087   
...        ...       ...             ...          ...            ...   
7634  1.252186 -1.375820        0.130360     1.979224      -1.948177   
7635  2.341911  1.204180        0.274557     1.473359      -1.023132   
7636 -0.241141  0.688180        0.454803    -0.044236       0.364435   
7637 -0.927265  0.172180        0.671099    -2.067695       0.364435   
7638 -0.039340 -0.859820       -0.698772     1.473359      -1.023132   

      Heart_Rate  Sleep_Hours  Exercise_Hours  Water_Intake  Stress_Level  \
0       2.004100     2.570772        0.091166      2.10113

# Applying DIfferent Models

## Logistic Regression

In [12]:
log_reg = LogisticRegression(
    penalty='l2',
    random_state=42,
    solver='lbfgs',
    max_iter=1000
)
log_reg.fit(x_train_scaled, y_train)
y_pred_lr = log_reg.predict(x_test_scaled)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Logistic Regression Recall:", recall_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


Logistic Regression Accuracy: 0.8136125654450261
Logistic Regression Recall: 0.8273092369477911
              precision    recall  f1-score   support

           0       0.81      0.80      0.80       914
           1       0.82      0.83      0.82       996

    accuracy                           0.81      1910
   macro avg       0.81      0.81      0.81      1910
weighted avg       0.81      0.81      0.81      1910



## kNN

In [13]:
k_values = [1, 3, 5, 7, 9, 11, 21, 50, 100, 110, 150, 200, 220, 250, 300, 340, 360, 400, 450, 500, 1000]

best_k = None
best_recall = 0

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(
        model,
        x_train_scaled,
        y_train,
        cv=5,
        scoring='recall'
    )

    mean_recall = scores.mean()

    print(f"k = {k}, Recall = {mean_recall:.4f}")

    if mean_recall > best_recall:
        best_recall = mean_recall
        best_k = k

print("Best k based on recall:", best_k)
print("Best recall:", best_recall)

k = 1, Recall = 0.8097
k = 3, Recall = 0.8581
k = 5, Recall = 0.8767
k = 7, Recall = 0.8807
k = 9, Recall = 0.8835
k = 11, Recall = 0.8923
k = 21, Recall = 0.9104
k = 50, Recall = 0.9134
k = 100, Recall = 0.9214
k = 110, Recall = 0.9229
k = 150, Recall = 0.9264
k = 200, Recall = 0.9282
k = 220, Recall = 0.9284
k = 250, Recall = 0.9305
k = 300, Recall = 0.9289
k = 340, Recall = 0.9310
k = 360, Recall = 0.9300
k = 400, Recall = 0.9327
k = 450, Recall = 0.9332
k = 500, Recall = 0.9342
k = 1000, Recall = 0.9355
Best k based on recall: 1000
Best recall: 0.9354763150760075


In [14]:
k_values = [1, 3, 5, 7, 9, 11, 21, 50, 100, 110, 150, 200, 220, 250, 300, 340, 360, 400, 450, 500, 1000]

best_k = None
best_f1 = 0

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)

    scores = cross_val_score(
        model,
        x_train_scaled,
        y_train,
        cv=5,
        scoring='f1'
    )

    mean_f1 = scores.mean()

    print(f"k = {k}, F1-score = {mean_f1:.4f}")

    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_k = k

print("\nBest k based on F1-score:", best_k)
print("Best F1-score:", best_f1)

k = 1, F1-score = 0.8159
k = 3, F1-score = 0.8560
k = 5, F1-score = 0.8725
k = 7, F1-score = 0.8786
k = 9, F1-score = 0.8809
k = 11, F1-score = 0.8883
k = 21, F1-score = 0.9016
k = 50, F1-score = 0.9060
k = 100, F1-score = 0.9049
k = 110, F1-score = 0.9052
k = 150, F1-score = 0.9022
k = 200, F1-score = 0.9001
k = 220, F1-score = 0.8995
k = 250, F1-score = 0.8982
k = 300, F1-score = 0.8951
k = 340, F1-score = 0.8946
k = 360, F1-score = 0.8930
k = 400, F1-score = 0.8919
k = 450, F1-score = 0.8897
k = 500, F1-score = 0.8889
k = 1000, F1-score = 0.8656

Best k based on F1-score: 50
Best F1-score: 0.9060022252424014


In [15]:
# Best k_neighbours = 50
knn = KNeighborsClassifier(
    n_neighbors=50,
    metric="euclidean"
)

knn.fit(x_train_scaled, y_train)

y_pred_knn = knn.predict(x_test_scaled)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))
print("KNN Recall:", recall_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

KNN Accuracy: 0.9073298429319372
KNN Recall: 0.9206827309236948
              precision    recall  f1-score   support

           0       0.91      0.89      0.90       914
           1       0.90      0.92      0.91       996

    accuracy                           0.91      1910
   macro avg       0.91      0.91      0.91      1910
weighted avg       0.91      0.91      0.91      1910



## Random Forest

In [16]:
param_grid = {
    'n_estimators': [50, 100, 150, 200, 300, 500]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='recall'
)

grid.fit(x_train, y_train)

print("Best n_estimators:", grid.best_params_)
print("Best score:", grid.best_score_)

# GridSearch CV evaluates on training dataset

Best n_estimators: {'n_estimators': 500}
Best score: 0.9480268342969553


In [18]:
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    random_state=42
)

rf.fit(x_train, y_train)

y_pred_rf = rf.predict(x_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest Recall:", recall_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# Here wwe have evaluated on tarining dataset

Random Forest Accuracy: 0.9397905759162304
Random Forest Recall: 0.9598393574297188
              precision    recall  f1-score   support

           0       0.95      0.92      0.94       914
           1       0.93      0.96      0.94       996

    accuracy                           0.94      1910
   macro avg       0.94      0.94      0.94      1910
weighted avg       0.94      0.94      0.94      1910



## Gradient Boosting

In [19]:
param_grid = {
    'n_estimators': [50, 100, 150, 200, 250],
    'learning_rate': [0.01, 0.05, 0.1, 0.18, 0.2],
    'max_depth': [3, 5, 7, 9]
}
grid = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='recall',   
    n_jobs=-1       
)

grid.fit(x_train, y_train)
print("Best parameters:", grid.best_params_)
print("Best score:", grid.best_score_)

Best parameters: {'learning_rate': 0.2, 'max_depth': 9, 'n_estimators': 150}
Best score: 0.9543054040591918


In [20]:
model = grid.best_estimator_

y_pred = model.predict(x_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.94      0.95       914
           1       0.94      0.96      0.95       996

    accuracy                           0.95      1910
   macro avg       0.95      0.95      0.95      1910
weighted avg       0.95      0.95      0.95      1910



In [21]:
gb = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.2,
    max_depth=9,
    random_state=42
)

gb.fit(x_train, y_train)

y_pred_gb = gb.predict(x_test)

print("Gradient Boosting Accuracy:", accuracy_score(y_test, y_pred_gb))
print("Gradient Boosting Recall:", recall_score(y_test, y_pred_gb))
print(classification_report(y_test, y_pred_gb))  

Gradient Boosting Accuracy: 0.949738219895288
Gradient Boosting Recall: 0.9618473895582329
              precision    recall  f1-score   support

           0       0.96      0.94      0.95       914
           1       0.94      0.96      0.95       996

    accuracy                           0.95      1910
   macro avg       0.95      0.95      0.95      1910
weighted avg       0.95      0.95      0.95      1910



## Voting Classifier

In [36]:
voting_clf = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, solver="liblinear")),
        ("knn", KNeighborsClassifier(n_neighbors=50)),
        ("rf", RandomForestClassifier(n_estimators=250, random_state=42))
    ],
    voting="soft"
)

voting_clf.fit(x_train_scaled, y_train)

y_pred_vote = voting_clf.predict(x_test_scaled)

print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred_vote))
print("Voting Classifier Recall:", recall_score(y_test, y_pred_vote))
print(classification_report(y_test, y_pred_vote))

Voting Classifier Accuracy: 0.9073298429319372
Voting Classifier Recall: 0.928714859437751
              precision    recall  f1-score   support

           0       0.92      0.88      0.90       914
           1       0.90      0.93      0.91       996

    accuracy                           0.91      1910
   macro avg       0.91      0.91      0.91      1910
weighted avg       0.91      0.91      0.91      1910



# Results

| Model                | Recall |
|----------------------|:------:|
| Logistic Regression  | 82,7%  |
| KNN                  | 92.0%  |
| Random Forest        | 95.9%  |
| Gradient Boosting    | 96.1%  |
| Voting Classifier    | 92.8% |

### Best Classifier that we should use for NovaGen(based on Recall) - Gradient Boosting with recall value of 96.1%